In [1]:
!pip install groq youtube-transcript-api chromadb sentence-transformers streamlit

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from groq import Groq
client=Groq(api_key="YOUR_GROQ_API_KEY_HERE")
response= client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"system",
            "content":"You are helpful assistant"
        },
        {
            "role":"user",
            "content":"What is retrieval augumented generation?"
        }
    ]
)
print(response.choices[0].message.content)

Retrieval-augmented generation (RAG) is a technique used in natural language processing (NLP) and artificial intelligence (AI) to improve the quality and informativeness of generated text. It combines the strengths of two approaches:

1. **Generative models**: These models, such as language models or sequence-to-sequence models, generate text from scratch based on the input prompt or context.
2. **Retrieval-based models**: These models retrieve relevant information from a database or a knowledge graph to answer a question or provide information on a specific topic.

In RAG, the generative model is augmented with a retrieval component that fetches relevant information from a database or knowledge graph. The retrieved information is then used to inform and guide the generative model to produce more accurate, informative, and context-specific text.

The RAG process typically involves the following steps:

1. **Input**: The user provides an input prompt or question.
2. **Retrieval**: The r

In [3]:
from groq import Groq
client=Groq(api_key="YOUR_GROQ_API_KEY_HERE")
response= client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"system",
            "content":"You are strict fast checker who finds answer only from the given conetext"
        },
        {
            "role":"user",
            "content":"""Content:
            RAG stands for Retrieval Augmented Generation. It is a technique where 
            an LLM is given external documents to answer questions instead of relying 
            on its training memory. This reduces hallucinations.
            Question:What is RAG?"""
        }
    ]
)
print(response.choices[0].message.content)

RAG stands for Retrieval Augmented Generation. It is a technique where an LLM is given external documents to answer questions instead of relying on its training memory.


In [4]:
from groq import Groq
client=Groq(api_key="YOUR_GROQ_API_KEY_HERE")
response= client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role":"system",
            "content":"You are strict fast checker who finds answer only from the given conetext"
        },
        {
            "role":"user",
            "content":"""Content:
            RAG stands for Retrieval Augmented Generation. It is a technique where 
            an LLM is given external documents to answer questions instead of relying 
            on its training memory. This reduces hallucinations.
            Question:What is Cyber Security?"""
        }
    ]
)
print(response.choices[0].message.content)

The given content does not provide information about Cyber Security. It only explains the concept of RAG (Retrieval Augmented Generation). Therefore, I cannot provide an answer to the question "What is Cyber Security?" based on the provided context.


## Observation:
-The system message controls how the LLM behaves
- Same model + same question + different system message = different answer
- Helpful assistant speaks from memory and can hallucinate
- Strict fact-checker only speaks from given context
- When context has no answer, strict checker refuses instead of guessing
- This refusal behavior is the core of GroundCheck's hallucination detection

## Fetching the transcipt!


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
video_id="LPZh9BOjkQs"
ytt_api = YouTubeTranscriptApi()
transcript_list = ytt_api.fetch(video_id)
for entry in transcript_list[:3]:
    print(entry)

FetchedTranscriptSnippet(text='Imagine you happen across a short movie script that', start=1.14, duration=2.836)
FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164)
FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58)


## Insepecting the Data


In [6]:
print(f"Total transcipts entries:{len(transcript_list)}")
print(f"Type transcipts_list:{type(transcript_list)}")
print(f"Type of one entry:{type(transcript_list[0])}")

Total transcipts entries:115
Type transcipts_list:<class 'youtube_transcript_api._transcripts.FetchedTranscript'>
Type of one entry:<class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>


## Joining into plain text

In [7]:
full_transcript=" ".join([entry.text for entry in transcript_list])
print(f"Total characters:{len(full_transcript)}")
print(f"Estimated Tokens:{len(full_transcript)/4:.0f}")
print(f"\n First 500 characters:\n")
print(full_transcript[:500])

Total characters:7524
Estimated Tokens:1881

 First 500 characters:

Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over 


## Proving a kind of problem:


In [8]:
print("__The problem__")
print(f"Full transcript tokens(estimated):~{len(full_transcript)/4:.0f}")
print(f"Safe token budget for transcript :~7692")
if len(full_transcript)/4>7692:
    print(f"\n Transcript is too long to send to LLM Mode:")
    print(f"We need to chunk it and send relevant pieces")
else:
    print(f"\n fits in the context but we can still chunk better retrieval accuracy")

__The problem__
Full transcript tokens(estimated):~1881
Safe token budget for transcript :~7692

 fits in the context but we can still chunk better retrieval accuracy


## Chunking 

In [9]:
def chunk_text(text,chunk_size=200,overlap=50):
    words=text.split()
    chunks=[]
    start=0
    while start <len(words):
        end=start+chunk_size
        chunk=" ".join(words[start:end])
        chunks.append(chunk)
        start+=chunk_size-overlap
    return chunks

## Running on transcript data

In [10]:
chunks=chunk_text(full_transcript)
print(f"Total chunks created:{len(chunks)}")
print(f"total per chunks :~{200/0.75:.0f}words=-267 tokens")
print(f"\n chunk 1")
print(chunks[0])
print(f"\n chunk 2")
print(chunks[1])
print(f"\n the last 50 words of chunk 1 vs first 50 words of chunk 2")
print(f"\n End of chunk 1:{' '.join(chunks[0].split()[-50])}")
print(f"\n End of chunk 2:{' '.join(chunks[1].split()[-50])}")

Total chunks created:9
total per chunks :~267words=-267 tokens

 chunk 1
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, you lay out some text that describes an interactio

## Understanding chunk 

In [11]:
print(f"type of chunks:{type(chunks)}")
print(f"type of one chunk:{type(chunks[0])}")
print(f"Number of chunks:{len(chunks)}")
print(f"Average chunk length :{sum(len(c)for c in chunks)//len(chunks)}")

type of chunks:<class 'list'>
type of one chunk:<class 'str'>
Number of chunks:9
Average chunk length :1081


## Embedding the Model:

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model done")
print(f"Model name:all-MiniLM-L6-v2")
test_embedding=embedding_model.encode("What is large language Model")
print(f"type :{type(test_embedding)}")
print(f"vector lenght:{len(test_embedding)}")
print(f"first five numbers:{test_embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model done
Model name:all-MiniLM-L6-v2
type :<class 'numpy.ndarray'>
vector lenght:384
first five numbers:[ 0.04031648 -0.09142097  0.00916874 -0.03463325 -0.01901193]


## Embedding all chunks

In [13]:
print("Embedding all chunks")
chunk_embeddings=embedding_model.encode(chunks)
print(f"Embedding Completed")
print(f"Number of embedding:{len(chunk_embeddings)}")
print(f"Shape of embeddings:{chunk_embeddings.shape}")
print(f"Each shape of vector is {chunk_embeddings.shape[1]}numbers")

Embedding all chunks
Embedding Completed
Number of embedding:9
Shape of embeddings:(9, 384)
Each shape of vector is 384numbers


## Storing in ChromaDB

In [14]:
import chromadb
chroma_client=chromadb.Client()
collection=chroma_client.create_collection(name="groundcheck")
collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=[f"chunk_{i}"for i in range (len(chunks))]
)
print(f"ChromaDb collection is done")
print(f"Collection Name:groundcheck")
print(f"Total documents stores :{collection.count()}")

ChromaDb collection is done
Collection Name:groundcheck
Total documents stores :9


## Semanctic Search:

In [15]:
question="How does large language model learn from the data!"
question_embedding=embedding_model.encode(question).tolist()
results=collection.query(
    query_embeddings=[question_embedding],
    n_results=2
)
print(f"Questions:{question}")
print(f"Most relevant chunk")
print(results['documents'][0][0])
print(f"\n Second Relevant chunk")
print(results['documents'][0][1])

Questions:How does large language model learn from the data!
Most relevant chunk
read non-stop 24-7, it would take over 2600 years. Larger models since then train on much, much more. You can think of training a little bit like tuning the dials on a big machine. The way that a language model behaves is entirely determined by these many different continuous values, usually called parameters or weights. Changing those parameters will change the probabilities that the model gives for the next word on a given input. What puts the large in large language model is how they can have hundreds of billions of these parameters. No human ever deliberately sets those parameters. Instead, they begin at random, meaning the model just outputs gibberish, but they're repeatedly refined based on many example pieces of text. One of these training examples could be just a handful of words, or it could be thousands, but in either case, the way this works is to pass in all but the last word from that example 

## Building retriver function:

In [16]:
def retrieve_relevant_chunks(question, collection, embedding_model, n_results=2):
    question_vector = embedding_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[question_vector],
        n_results=n_results
    )
    chunks_found = results['documents'][0]
    
    return chunks_found
test_question = "What is a large language model?"
retrieved = retrieve_relevant_chunks(
    test_question, 
    collection, 
    embedding_model
)
print(f"Question: {test_question}")
print(f"Chunks retrieved: {len(retrieved)}")
print(f"\nChunk 1 preview: {retrieved[0][:200]}...")

Question: What is a large language model?
Chunks retrieved: 2

Chunk 1 preview: read non-stop 24-7, it would take over 2600 years. Larger models since then train on much, much more. You can think of training a little bit like tuning the dials on a big machine. The way that a lang...


## Building augmented prompt:

In [18]:
def build_augmented_prompt(question, chunks):
    context = "\n\n".join([
        f"[Chunk {i+1}]:\n{chunk}" 
        for i, chunk in enumerate(chunks)
    ])
    augmented_prompt = f"""You are GroundCheck — an AI assistant that answers 
questions very strictly based on the provided context only.

CONTEXT FROM VIDEO:
{context}

QUESTION:
{question}

INSTRUCTIONS:
- Answer only using information found in the context above
- If the answer is not in the context, say exactly: 
  "I could not find this in the video."
- Do not use your training knowledge
- Keep your answer clear and concise
- After your answer, list which chunk number(s) you used
"""
    
    return augmented_prompt
test_chunks = retrieve_relevant_chunks(
    "What is a large language model?",
    collection,
    embedding_model
)
prompt = build_augmented_prompt(
    "What is a large language model?", 
    test_chunks
)
print(" AUGMENTED PROMPT PREVIEW ")
print(prompt)
print(f"\nTotal prompt characters: {len(prompt)}")
print(f"Estimated prompt tokens: {len(prompt)/4:.0f}")

 AUGMENTED PROMPT PREVIEW 
You are GroundCheck — an AI assistant that answers 
questions very strictly based on the provided context only.

CONTEXT FROM VIDEO:
[Chunk 1]:
read non-stop 24-7, it would take over 2600 years. Larger models since then train on much, much more. You can think of training a little bit like tuning the dials on a big machine. The way that a language model behaves is entirely determined by these many different continuous values, usually called parameters or weights. Changing those parameters will change the probabilities that the model gives for the next word on a given input. What puts the large in large language model is how they can have hundreds of billions of these parameters. No human ever deliberately sets those parameters. Instead, they begin at random, meaning the model just outputs gibberish, but they're repeatedly refined based on many example pieces of text. One of these training examples could be just a handful of words, or it could be thousands, but

## Building QA function:

In [29]:
def groundcheck_answer(question,collection,embedding_model,client):
    relevant_chunks=retrieve_relevant_chunks(
        question,collection,embedding_model
    )
    prompt=build_augmented_prompt(question,relevant_chunks)
    response=client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
    {
    "role":"system",
        "content":""" You are groundcheck.you are very strict fast-checker.you only answer from the previous context.
    You never use outside knowledge """
    },
    {
        "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1 
    )
    answer = response.choices[0].message.content
    return {
        "question": question,
        "answer": answer,
        "chunks_used": relevant_chunks
    }
print("GroundCheck QA function done")

GroundCheck QA function done


In [30]:
result=groundcheck_answer(
    question="How does large language model learn from data",
    collection=collection,
    embedding_model=embedding_model,
    client=client
    )
print("="*50)
print(f"question:\n{result['question']}")
print("="*50)
print(f"\nGround check answer:\n{result['answer']}")
print("="*50)
print(f"\n chunks in this answer")
for i,chunk in enumerate(result['chunks_used']):
    print(f"\n[chunk{i+1}preview]:{chunk[:500]}")

question:
How does large language model learn from data

Ground check answer:
A large language model learns from data by having its parameters refined based on many example pieces of text. The model is given all but the last word from an example, and its prediction is compared to the true last word. An algorithm called backpropagation is used to tweak the parameters, making the model more likely to choose the true last word and less likely to choose others.

I used Chunk 1 and Chunk 2.

 chunks in this answer

[chunk1preview]:read non-stop 24-7, it would take over 2600 years. Larger models since then train on much, much more. You can think of training a little bit like tuning the dials on a big machine. The way that a language model behaves is entirely determined by these many different continuous values, usually called parameters or weights. Changing those parameters will change the probabilities that the model gives for the next word on a given input. What puts the large in large lan

## Hallucination:

In [32]:
result_outside=groundcheck_answer(
    question="What is networking",
    collection=collection,
    embedding_model=embedding_model,
    client=client
)
print("="*50)
print(f"Question:\n{result_outside['question']}")
print("="*50)
print(f"\n grounc check answer:\n{result_outside['answer']}")

Question:
What is networking

 grounc check answer:
I could not find this in the video.
Chunk 1, Chunk 2


## Hallucination detecting :

In [39]:
def detect_hallucination(question, answer, chunks):
    context = "\n\n".join([
        f"[Chunk {i+1}]:\n{chunk}" 
        for i, chunk in enumerate(chunks)
    ])
    eval_prompt = f"""You are a very strict evaluation engine for an AI system  called GroundCheck. Your job is to evaluate whether an AI answer is faithful to its source context or contains hallucinations.
SOURCE CONTEXT:
{context}
QUESTION THAT WAS ASKED:
{question}
AI ANSWER TO EVALUATE:
{answer}

Evaluate the answer on these three criteria and respond in 
exactly this format — nothing else:

FAITHFULNESS: [score 1-10] | [one sentence explanation]
COMPLETENESS: [score 1-10] | [one sentence explanation]  
HALLUCINATION_DETECTED: [YES or NO] | [one sentence explanation]
OVERALL_VERDICT: [TRUSTED or FLAGGED] | [one sentence explanation]
"""
    eval_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a precise AI evaluation engine. 
                You respond only in the exact format requested. 
                You are brutally honest about hallucinations."""
            },
            {
                "role": "user",
                "content": eval_prompt
            }
        ],
        temperature=0.0
    )
    evaluation = eval_response.choices[0].message.content
    return evaluation
print("Hallucination detect is ready")

Hallucination detect is ready


## Running the detect on answer:

In [40]:
good_result=groundcheck_answer(
    question="How does large language model learn from the data",
    collection=collection,
    embedding_model=embedding_model,
    client=client
    )
print("Answer from grouncheck")
print(good_result['answer'])
print("\n"+"="*50)
print("Hallucination detect is:")
print("="*50)
evaluation =detect_hallucination(
    question=good_result['question'],
    answer=good_result['answer'],
    chunks=good_result['chunks_used']
)
print(evaluation)

Answer from grouncheck
The large language model learns from the data by having its parameters refined based on many example pieces of text. The model is given all but the last word from an example, and its prediction is compared to the true last word. An algorithm called backpropagation is used to tweak the parameters, making the model more likely to choose the true last word and less likely to choose others.

I used Chunk 1 and Chunk 2.

Hallucination detect is:
FAITHFULNESS: 9 | The answer accurately reflects the process of how a large language model learns from data as described in the source context.
COMPLETENESS: 8 | The answer covers the key points of the learning process but lacks some details, such as the scale of computation involved in training.
HALLUCINATION_DETECTED: NO | The answer does not introduce any information not present in the source context.
OVERALL_VERDICT: TRUSTED | The answer is a reliable and accurate representation of how a large language model learns from da

## detection of suspicious:

In [42]:
suspicious_result=groundcheck_answer(
    question="what are all types of neural network",
    collection=collection,
    embedding_model=embedding_model,
    client=client
)
print("Answer from groundcheck")
print(suspicious_result['answer'])
print("\n"+"="*50)
print("Hallucination detect")
print("="*50)
suspicious_result=detect_hallucination(
    question=suspicious_result['question'],
    answer=suspicious_result['answer'],
    chunks=suspicious_result['chunks_used']
)

print(suspicious_result)

Answer from groundcheck
There are two types of neural network operations mentioned: 
1. backpropagation 
2. feed-forward neural network.

I used chunk(s): 1, 2

Hallucination detect
FAITHFULNESS: 8 | The answer accurately identifies two types of neural network operations mentioned in the source context, but slightly misrepresents backpropagation as a type of neural network operation.
COMPLETENESS: 2 | The answer is incomplete because it only mentions two specific types of neural network operations and does not provide a comprehensive list of all types of neural networks.
HALLUCINATION_DETECTED: NO | The answer does not introduce any information not present in the source context, avoiding hallucination.
OVERALL_VERDICT: FLAGGED | The answer is flagged due to its incompleteness and potential for misleading readers about the types of neural networks.


In [49]:
def groundcheck_full(question):
    print(f"\n{'='*50}")
    print(f"GROUNDCHECK ANALYSIS")
    print(f"{'='*50}")
    print(f"Question: {question}")
    print(f"{'='*50}\n")
    result = groundcheck_answer(
        question=question,
        collection=collection,
        embedding_model=embedding_model,
        client=client
    )
    print(f"ANSWER:\n{result['answer']}\n")
    print(f"{'='*50}")
    print(f"RELIABILITY REPORT:")
    print(f"{'='*50}")
    evaluation = detect_hallucination(
        question=result['question'],
        answer=result['answer'],
        chunks=result['chunks_used']
    )
    print(evaluation)
    print(f"{'='*50}\n")
    return {
        "question": question,
        "answer": result['answer'],
        "evaluation": evaluation,
        "chunks": result['chunks_used']
    }
print("done")

done


In [50]:
groundcheck_full(
    "what is backpropagation and hw it works in LLM"
)
groundcheck_full(
    "What is the best GPU for training language models?"
)
groundcheck_full(
    "Who is the president of the Iran?"
)


GROUNDCHECK ANALYSIS
Question: what is backpropagation and hw it works in LLM

ANSWER:
Backpropagation is an algorithm used to tweak all of the parameters in a large language model in such a way that it makes the model a little more likely to choose the true last word and a little less likely to choose all the others. It works by passing in all but the last word from a training example into the model, comparing the prediction with the true last word, and then tweaking the parameters based on this comparison.

I used chunk 1 and chunk 2.

RELIABILITY REPORT:
FAITHFULNESS: 9 | The answer accurately reflects the information provided in the source context about backpropagation, with a minor omission of details.
COMPLETENESS: 8 | The answer provides a clear and concise explanation of backpropagation, but lacks some additional context and details present in the source chunks.
HALLUCINATION_DETECTED: NO | The answer does not introduce any information not present in the source context, sticki

{'question': 'Who is the president of the Iran?',
 'answer': 'I could not find this in the video.\nChunk 1, Chunk 2',
 'evaluation': 'FAITHFULNESS: 10 | The answer is faithful to the source context as it does not provide any information that contradicts or misrepresents the content of the chunks.\nCOMPLETENESS: 8 | The answer is incomplete because it does not provide a direct response to the question asked, but it is reasonable given the context provided.\nHALLUCINATION_DETECTED: NO | No hallucination is detected because the answer does not introduce any information that is not present in the source context or is not a reasonable inference from it.\nOVERALL_VERDICT: TRUSTED | The answer is trusted because it is a reasonable response given the context, even though it does not directly answer the question, and it does not introduce any hallucinated information.',
 'chunks': ["that are optimized for running many operations in parallel, known as GPUs. However, not all language models can b